# FER2013 Data Exploration

Use this notebook in Colab or Kaggle to inspect the FER2013 image-folder dataset before training. The goal is to confirm split sizes, class imbalance, and sample image quality.


## 1. Setup

Upload or mount the repository, then update `PROJECT_ROOT` and `DATA_DIR` if needed.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

# Local path after extracting Kaggle data:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'fer2013_images'

# Kaggle example, uncomment and edit if needed:
# DATA_DIR = Path('/kaggle/input/fer2013')

DATA_DIR


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image

from fer_project.data import EMOTION_LABELS, folder_class_to_fer_label


## 2. Load FER2013 Image Folders


In [ ]:
rows = []
for split in ['train', 'test']:
    split_dir = DATA_DIR / split
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        label = folder_class_to_fer_label(class_dir.name)
        image_paths = list(class_dir.glob('*'))
        rows.append({
            'split': split,
            'emotion': label,
            'emotion_name': EMOTION_LABELS[label],
            'folder': class_dir.name,
            'count': len(image_paths),
        })

counts = pd.DataFrame(rows).sort_values(['split', 'emotion'])
counts


In [ ]:
print(counts.groupby('split')['count'].sum())
counts.pivot_table(index='emotion_name', columns='split', values='count', fill_value=0)


## 3. Class Balance

In [ ]:
plt.figure(figsize=(9, 4))
sns.barplot(data=counts, x='emotion_name', y='count', hue='split', order=[EMOTION_LABELS[i] for i in range(7)])
plt.xticks(rotation=30)
plt.tight_layout()


## 4. Image Samples

In [ ]:
fig, axes = plt.subplots(7, 6, figsize=(8, 9))
for label, emotion in EMOTION_LABELS.items():
    class_dir = DATA_DIR / 'train' / emotion
    samples = sorted(class_dir.glob('*'))[:6]
    for ax, image_path in zip(axes[label], samples):
        ax.imshow(Image.open(image_path), cmap='gray')
        ax.axis('off')
    axes[label][0].set_ylabel(emotion)
plt.tight_layout()


## Notes for Report

Record class imbalance, visible label ambiguity, and any low-quality images. This supports the instructor-requested error analysis.